# DuckDB Storage Browser (macOS)

This notebook scans `~/Library/Application Support/com.adsb.aircraft-tracker/` for DuckDB database files, lists tables, and previews a few rows from each table.

Notes:
- It opens databases in **read-only** mode (no writes).
- If your app stores the DB with a different extension, adjust the patterns in the scan cell.

In [ ]:
# If you don't have duckdb installed in this Jupyter environment, run:
%pip install duckdb numpy pandas 

import os
from pathlib import Path
import glob

import duckdb

In [ ]:
# Root directory where the Tauri app stores data (default)
ROOT_DIR = os.path.expanduser('~/Library/Application Support/com.adsb.aircraft-tracker/')
ROOT_DIR

In [ ]:
def find_duckdb_files(root_dir: str) -> list[Path]:
    root = Path(os.path.expanduser(root_dir)).resolve()
    patterns = [
        str(root / '*.duckdb'),
        str(root / '*.db'),
        str(root / '**' / '*.duckdb'),
        str(root / '**' / '*.db'),
    ]

    files: list[Path] = []
    for pat in patterns:
        for p in glob.glob(pat, recursive=True):
            try:
                path = Path(p).resolve()
            except FileNotFoundError:
                continue
            if path.is_file():
                files.append(path)

    # De-dup while preserving order
    seen: set[Path] = set()
    out: list[Path] = []
    for f in files:
        if f not in seen:
            seen.add(f)
            out.append(f)
    return out

db_files = find_duckdb_files(ROOT_DIR)
db_files

## List tables and preview rows
Adjust `PREVIEW_LIMIT` to show more/less rows per table.

In [ ]:
PREVIEW_LIMIT = 10

def list_user_tables(con: duckdb.DuckDBPyConnection):
    return con.execute("""
        SELECT table_schema, table_name, table_type
        FROM information_schema.tables
        WHERE table_schema NOT IN ('information_schema', 'pg_catalog')
        ORDER BY table_schema, table_name
    """).fetchall()

def preview_table(con: duckdb.DuckDBPyConnection, schema: str, table: str, limit: int):
    # Quote identifiers safely
    ident = f'"{schema}"."{table}"'
    return con.execute(f'SELECT * FROM {ident} LIMIT {int(limit)}').df()

if not db_files:
    raise FileNotFoundError(f'No .duckdb/.db files found under: {ROOT_DIR}')

for db_path in db_files:
    print('' + '=' * 80)
    print(db_path)
    print('=' * 80)
    
    try:
        con = duckdb.connect(database=str(db_path), read_only=True)
    except Exception as e:
        print(f'Failed to open DB: {e}')
        continue

    try:
        tables = list_user_tables(con)
        if not tables:
            print('(no user tables found)')
            continue
        
        print('Tables:')
        for schema, name, ttype in tables:
            print(f'  - {schema}.{name} [{ttype}]')
        
        for schema, name, ttype in tables:
            # Only preview base tables by default; comment this out to preview views too
            if ttype != 'BASE TABLE':
                continue
            print(f'Preview: {schema}.{name} (limit {PREVIEW_LIMIT}) --')
            try:
                df = preview_table(con, schema, name, PREVIEW_LIMIT)
                if df.shape[0] == 0:
                    print('(no rows)')
                else:
                    display(df)
            except Exception as e:
                print(f'Failed to query: {e}')
    finally:
        con.close()